# Audio Transcription with Voxtral and Mistral AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mistralai/cookbook/blob/main/mistral/audio/audio_transcription_voxtral.ipynb)

This notebook demonstrates how to use Mistral's audio capabilities for transcription tasks using the Mistral Python client. We cover:

- Basic audio transcription from a file and from a URL
- Transcription with word-level and segment-level timestamps
- Speaker diarization
- Context biasing for domain-specific vocabulary
- Multimodal chat with audio input

### Models Used
- Voxtral Mini (`voxtral-mini-latest`) for transcription

### Supported Languages
Voxtral supports 13 languages: English, Chinese, Hindi, Spanish, Arabic, French, Portuguese, Russian, German, Japanese, Korean, Italian, and Dutch.

### Supported Formats
`.mp3`, `.wav`, `.m4a`, `.flac`, `.ogg` (up to 1GB per file, up to 3 hours per request).

## Setup

First, install the Mistral AI Python client.

In [ ]:
%%capture
!pip install mistralai>=1.6.0

## Initialize Client

Set up the Mistral client with your API key. You can create an API key on the [Mistral Console](https://console.mistral.ai/api-keys/).

In [ ]:
import os
from mistralai import Mistral

api_key = os.environ["MISTRAL_API_KEY"]
client = Mistral(api_key=api_key)

model = "voxtral-mini-latest"

## Basic Audio Transcription

### From a URL

The simplest way to transcribe audio is by passing a URL to the audio file.

In [ ]:
# Transcribe audio from a URL
transcription_response = client.audio.transcriptions.complete(
    model=model,
    file_url="https://docs.mistral.ai/audio/obama.mp3",
)

print(transcription_response.text)

### From a Local File

You can also transcribe a local audio file by reading its content.

In [ ]:
%%capture
!wget -q -O sample_audio.mp3 https://docs.mistral.ai/audio/obama.mp3

In [ ]:
# Transcribe a local audio file
with open("sample_audio.mp3", "rb") as f:
    transcription_response = client.audio.transcriptions.complete(
        model=model,
        file={
            "content": f,
            "file_name": "sample_audio.mp3",
        },
    )

print(transcription_response.text)

### Specifying the Language

Providing the language code can boost transcription accuracy.

In [ ]:
# Transcribe with explicit language hint
transcription_response = client.audio.transcriptions.complete(
    model=model,
    file_url="https://docs.mistral.ai/audio/obama.mp3",
    language="en",
)

print(transcription_response.text)

## Transcription with Timestamps

Voxtral can produce precise timestamps at segment and word level, useful for subtitle generation and audio alignment.

### Segment-Level Timestamps

In [ ]:
import json

# Transcribe with segment-level timestamps
transcription_response = client.audio.transcriptions.complete(
    model=model,
    file_url="https://docs.mistral.ai/audio/obama.mp3",
    timestamp_granularities=["segment"],
)

print(f"Full text: {transcription_response.text[:200]}...\n")

# Display segments with timestamps
if transcription_response.segments:
    for segment in transcription_response.segments[:5]:
        print(json.dumps(segment.model_dump(), indent=2))

### Word-Level Timestamps

In [ ]:
# Transcribe with word-level timestamps
transcription_response = client.audio.transcriptions.complete(
    model=model,
    file_url="https://docs.mistral.ai/audio/obama.mp3",
    timestamp_granularities=["word"],
)

print(f"Full text: {transcription_response.text[:200]}...\n")

# Display first 10 word-level timestamps
if transcription_response.segments:
    for segment in transcription_response.segments[:1]:
        if hasattr(segment, 'words') and segment.words:
            for word in segment.words[:10]:
                print(json.dumps(word.model_dump(), indent=2))

## Speaker Diarization

Voxtral can automatically identify different speakers in the audio, which is useful for meeting transcriptions and multi-party call processing.

In [ ]:
# Transcribe with speaker diarization
transcription_response = client.audio.transcriptions.complete(
    model=model,
    file_url="https://docs.mistral.ai/audio/obama.mp3",
    diarize=True,
    timestamp_granularities=["segment"],
)

print(f"Full text: {transcription_response.text[:200]}...\n")

# Display segments with speaker labels
if transcription_response.segments:
    for segment in transcription_response.segments[:5]:
        print(json.dumps(segment.model_dump(), indent=2))

## Context Biasing

You can provide up to 100 words or phrases to guide the model toward correct spellings of names, technical terms, or domain-specific vocabulary. This is particularly useful for proper nouns, acronyms, or specialized jargon.

In [ ]:
# Transcribe with context biasing for domain-specific terms
transcription_response = client.audio.transcriptions.complete(
    model=model,
    file_url="https://docs.mistral.ai/audio/obama.mp3",
    context_bias=(
        "Chicago,Joplin,Boston,Charleston,farewell_address,"
        "self-government,citizenship,democracy,American_people,"
        "cancer_survivors,affordable_health_care,wounded_warriors,"
        "refugees,elected_officials,American_spirit"
    ),
)

print(transcription_response.text[:500])

## Multimodal Chat with Audio Input

Beyond pure transcription, Voxtral can be used in chat completions to understand and reason about audio content. This enables use cases like audio summarization, question answering about audio, and audio-based instructions.

In [ ]:
import base64

# Read and encode the audio file in base64
with open("sample_audio.mp3", "rb") as f:
    audio_base64 = base64.b64encode(f.read()).decode("utf-8")

# Use audio in a chat completion for summarization
chat_response = client.chat.complete(
    model=model,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_audio",
                    "input_audio": audio_base64,
                },
                {
                    "type": "text",
                    "text": "Summarize the key points of this audio recording.",
                },
            ],
        }
    ],
)

print(chat_response.choices[0].message.content)

In [ ]:
# Ask a specific question about the audio content
chat_response = client.chat.complete(
    model=model,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_audio",
                    "input_audio": audio_base64,
                },
                {
                    "type": "text",
                    "text": "What language is this audio in? What is the overall tone and sentiment?",
                },
            ],
        }
    ],
)

print(chat_response.choices[0].message.content)

## Usage Information

Each transcription response includes usage information that can help track API consumption.

In [ ]:
# Check usage from the last transcription
transcription_response = client.audio.transcriptions.complete(
    model=model,
    file_url="https://docs.mistral.ai/audio/obama.mp3",
)

print(f"Model: {transcription_response.model}")
print(f"Language: {transcription_response.language}")
print(f"Usage: {transcription_response.usage}")

## Best Practices

1. **Specify language**: Always provide the `language` parameter when you know the audio language to improve accuracy.
2. **Use context biasing**: For domain-specific content, provide relevant terms via `context_bias` to improve recognition of proper nouns, technical terms, and acronyms.
3. **Enable diarization** for multi-speaker audio: Set `diarize=True` when transcribing meetings, interviews, or conversations.
4. **Choose appropriate timestamps**: Use `segment` granularity for general alignment and `word` granularity for subtitle generation.
5. **File size limits**: Keep audio files under 1GB and under 3 hours per request.
6. **Supported formats**: Use `.mp3`, `.wav`, `.m4a`, `.flac`, or `.ogg` formats.
7. **Multimodal chat**: Use `chat.complete()` with `input_audio` content type when you need the model to reason about audio content, not just transcribe it.

## Resources

- [Mistral Audio Documentation](https://docs.mistral.ai/capabilities/audio/)
- [Mistral API Reference](https://docs.mistral.ai/api/)
- [Mistral Python Client](https://github.com/mistralai/client-python)
- [Voxtral on Hugging Face](https://huggingface.co/mistralai)